# ClimaCity Paris -- Jour 2
## Spark SQL, Delta Lake et Structured Streaming

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Durée** : 1 journée (6 heures effectives)  
**Prérequis** : Avoir complété le Jour 1 -- la table `disponibilite_consolidee.parquet` doit être présente dans `data/output/`

---

Ce notebook couvre l'intégralité du Jour 2 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : Spark SQL et l'API de fenêtrage analytique, puis Delta Lake
  pour la persistance transactionnelle (écriture, time-travel, `MERGE INTO`).
- **Partie 2 -- Après-midi (3 h)** : Structured Streaming -- connexion à un flux simulé
  de mises à jour de stations, agrégations sur fenêtres glissantes, gestion des données
  tardives (late data) et déclenchement d'alertes.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne à compléter.  
> Les cellules `# [CORRECTION]` proposent une solution -- ne les regardez qu'après avoir tenté.


---
## Section 0 -- Configuration

Même structure de chemins qu'au Jour 1. La table consolidée produite hier est le
point de départ de toutes les analyses.


In [1]:
from pathlib import Path
import time

# ── Chemins ─────────────────────────────────────────────────────────────────
DATA_DIR           = Path("../data")
OUTPUT_DIR         = DATA_DIR / "output"
VELIB_CONSOLIDE    = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE   = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES      = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR  = OUTPUT_DIR / "stream_input"    # répertoire surveillé par Spark
STREAM_CHECKPOINT  = OUTPUT_DIR / "checkpoints"

for p in [VELIB_CONSOLIDE]:
    assert p.exists(), f"Fichier manquant : {p} -- relancez le Jour 1"

for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Paramètres ───────────────────────────────────────────────────────────────
APP_NAME      = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8
SEED          = 42


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Delta Lake requiert le package delta-spark
# Vérifiez que votre environnement conda l'inclut avant de démarrer.
spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.driver.memory", "6g")
    # Horodatages du projet en UTC : même fuseau que le carnet 1 (les colonnes calendaires
    # heure / jour_sem de la table consolidée sont, elles, déjà en heure de Paris).
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- Delta Lake activé")
print(f"Spark UI : http://localhost:4040")

Spark 3.5.3 -- Delta Lake activé
Spark UI : http://localhost:4040


In [3]:
# Chargement de la table consolidée produite au Jour 1
df = spark.read.parquet(str(VELIB_CONSOLIDE))
df.cache()
df.count()   # force la mise en cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()


Table consolidée : 10,769,262 lignes  |  20 colonnes
root
 |-- station_id: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- horodatage: timestamp (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- statut: string (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- est_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: double (nullable = true)
 |-- vent_kmh: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)



---
# PARTIE 1 -- Spark SQL (matin)

## 1.1 Vues temporaires et premières requêtes SQL

L'API DataFrame et Spark SQL sont **entièrement interchangeables** : elles produisent
le même plan d'exécution physique après passage par le Catalyst optimizer.
Le choix entre les deux est une question de lisibilité et d'habitude.

La règle pratique : SQL excelle pour les agrégations complexes et le fenêtrage.
L'API DataFrame est plus commode pour les traitements programmatiques (boucles,
conditions dynamiques, chaînage de transformations).


In [4]:
# Enregistrement des vues temporaires
# Une vue temporaire n'existe que pour la durée de la session Spark.
# Elle ne copie pas les données -- c'est un alias sur le DataFrame.
df.createOrReplaceTempView("disponibilite")

# Vérification
spark.sql("SHOW VIEWS").show()


+---------+-------------+-----------+
|namespace|     viewName|isTemporary|
+---------+-------------+-----------+
|         |disponibilite|       true|
+---------+-------------+-----------+



In [5]:
# Première requête : distribution des statuts par arrondissement :
# Nom de 'snapshots', taux d'occuoation moyen, écarrt-type de l'occupation
spark.sql("""
-- GROUP BY sur (arrondissement, statut) : une ligne par couple, avec trois agrégats calculés
-- en une seule passe sur les données. STDDEV mesure la dispersion : un écart-type élevé
-- indique un arrondissement où les stations passent de vides à saturées.
SELECT code_arr,
       statut,
       COUNT(*)                        AS nb_snapshots,
       ROUND(AVG(taux_occupation), 3)  AS taux_moyen,
       ROUND(STDDEV(taux_occupation), 3) AS ecart_type
FROM disponibilite
GROUP BY code_arr, statut
ORDER BY code_arr, statut
""").show(40)

+--------+------+------------+----------+----------+
|code_arr|statut|nb_snapshots|taux_moyen|ecart_type|
+--------+------+------------+----------+----------+
|       1|normal|      163015|     0.521|     0.232|
|       1| plein|       20349|      0.95|     0.031|
|       1|  vide|        8380|     0.056|     0.029|
|       2|normal|      171633|     0.446|      0.23|
|       2| plein|       10612|     0.954|     0.031|
|       2|  vide|       20988|     0.052|     0.027|
|       3|normal|      100439|     0.554|      0.21|
|       3| plein|        7703|     0.947|      0.03|
|       3|  vide|        1503|     0.058|     0.029|
|       4|normal|      180346|     0.531|     0.222|
|       4| plein|       20556|     0.952|     0.031|
|       4|  vide|        6011|     0.059|     0.028|
|       5|normal|      231952|     0.481|     0.243|
|       5| plein|       26603|     0.953|     0.029|
|       5|  vide|       31751|      0.05|     0.029|
|       6|normal|      212594|     0.475|     

In [6]:
# Les fonctions temporelles SQL sont disponibles directement
# Taux moyen d'occupation, nombre de snapsohts, nombre de stations par heure
spark.sql("""
-- `heure` est l'heure LOCALE de Paris (calculée au carnet 1) : on lit donc directement
-- le rythme de la journée (creux nocturne, pointes de 8 h et 18 h).
SELECT heure,
       ROUND(AVG(taux_occupation), 3) AS taux_moyen,
       COUNT(*)                       AS nb_snapshots,
       COUNT(DISTINCT station_id)     AS nb_stations
FROM disponibilite
GROUP BY heure
ORDER BY heure
""").show(24)

+-----+----------+------------+-----------+
|heure|taux_moyen|nb_snapshots|nb_stations|
+-----+----------+------------+-----------+
|    0|     0.393|      494324|       1382|
|    1|      0.39|      224463|       1382|
|    2|     0.394|      210811|       1382|
|    3|     0.397|      398427|       1382|
|    4|     0.398|      451779|       1382|
|    5|     0.396|      418908|       1382|
|    6|     0.397|      516155|       1382|
|    7|     0.387|      432640|       1382|
|    8|     0.366|      514754|       1382|
|    9|     0.365|      450439|       1382|
|   10|     0.368|      473661|       1382|
|   11|      0.36|      507878|       1382|
|   12|     0.344|      570954|       1382|
|   13|     0.341|      398373|       1383|
|   14|     0.347|      499698|       1383|
|   15|     0.345|      550423|       1382|
|   16|      0.34|      492800|       1382|
|   17|     0.331|      294260|       1383|
|   18|     0.338|      386015|       1382|
|   19|     0.344|      380624| 

---
## 1.2 Questions métier -- Requêtes analytiques

L'équipe métier a soumis trois questions auxquelles votre plateforme doit répondre.
Nous allons les traiter une par une avec Spark SQL.


### Question 1 : Ruptures en heure de pointe matinale

> Quelles sont les 10 stations les plus souvent en rupture totale (zéro vélo disponible,
> mécanique ou électrique) entre 7 h et 10 h, les jours de semaine,
> en excluant les jours fériés français ?

Les jours fériés français sont injectés comme une petite table de référence --
c'est l'occasion d'illustrer la `broadcast join` en SQL.


In [7]:
# Table de jours fériés (2020-2021) -- injectée en broadcast
# Source : légifrance.gouv.fr
# ADAPTATION : le modèle listait 2022-2023 ; nos données couvrent 26/11/2020 -> 09/04/2021.
# On liste les jours fériés français de cette période élargie (le 05/04/2021 est le lundi de
# Pâques ; le 25/12/2020 et le 01/01/2021 tombent un vendredi : ils exercent bien le filtre).
jours_feries = spark.createDataFrame([
    ("2020-11-01",), ("2020-11-11",), ("2020-12-25",),
    ("2021-01-01",), ("2021-04-05",),
], ["date_ferie"])

jours_feries = jours_feries.withColumn(
    "date_ferie", F.to_date("date_ferie", "yyyy-MM-dd")
)
jours_feries.createOrReplaceTempView("jours_feries")

print(f"{jours_feries.count()} jours fériés enregistrés (2020-2021)")

5 jours fériés enregistrés (2020-2021)


In [8]:
# Identifier les 10 stations Vélib' les plus en rupture de stock pendant les heures de pointe matinales, en excluant les jours fériés.
# On ne s'intéresse qu'aux stations trè!s fréquentées,ayant plus de 100 observations (snapshots)
# -> Id de la station, noù de la station, arroondissement, nombre d'observations (snapshots)
df_q1 = spark.sql("""
-- Logique de la requête :
--  * BROADCAST(f) : Spark copie la table des jours fériés (5 lignes) sur chaque exécuteur, ce
--    qui évite tout shuffle de la grande table `disponibilite` pour la jointure.
--  * LEFT ANTI JOIN : garde les relevés dont la date N'EST PAS dans les jours fériés
--    (un NOT IN se comporterait mal avec des NULL).
--  * Je compare la date fériée à la date LOCALE (fuseau de Paris) et je laisse la date UTC de côté.
--  * heure BETWEEN 7 AND 9 = de 7 h 00 à 9 h 59 (heure locale) ; NOT est_weekend = jour de semaine.
--  * Rupture = aucun vélo, mécanique comme électrique.
--  * HAVING COUNT(*) > 100 : écarte les stations trop peu observées, dont un taux de rupture
--    serait statistiquement peu fiable.
SELECT /*+ BROADCAST(f) */
       d.station_id,
       d.nom_station,
       d.code_arr,
       COUNT(*)                                                          AS nb_snapshots,
       SUM(CASE WHEN d.velos_meca + d.velos_elec = 0 THEN 1 ELSE 0 END)  AS nb_ruptures,
       ROUND(AVG(CASE WHEN d.velos_meca + d.velos_elec = 0 THEN 1.0 ELSE 0.0 END), 3) AS taux_rupture
FROM disponibilite d
LEFT ANTI JOIN jours_feries f
       ON to_date(from_utc_timestamp(d.horodatage, 'Europe/Paris')) = f.date_ferie
WHERE d.heure BETWEEN 7 AND 9
  AND NOT d.est_weekend
GROUP BY d.station_id, d.nom_station, d.code_arr
HAVING COUNT(*) > 100
ORDER BY nb_ruptures DESC, taux_rupture DESC
LIMIT 10
""")

df_q1.show(truncate=False)

+----------+------------------------------------+--------+------------+-----------+------------+
|station_id|nom_station                         |code_arr|nb_snapshots|nb_ruptures|taux_rupture|
+----------+------------------------------------+--------+------------+-----------+------------+
|2282      |Square de la rue Pixérécourt        |20      |714         |202        |0.283       |
|1924      |Piat - Parc de Belleville           |20      |714         |197        |0.276       |
|1571      |Hôtel de ville de Fontenay-sous-Bois|41      |714         |194        |0.272       |
|2380      |Villiers de l'Isle Adam - Pyrénées  |20      |714         |194        |0.272       |
|1001      |11 Novembre 1918 - 8 Mai 1945       |45      |714         |182        |0.255       |
|1791      |Maréchal Joffre - Verdun            |41      |714         |171        |0.240       |
|2085      |Porte de Ménilmontant               |20      |714         |167        |0.234       |
|2115      |Pyrénées - Ménilmo

### Question 2 : Impact de la pluie sur le taux d'occupation

> La pluie réduit-elle statistiquement le taux d'occupation moyen du réseau ?
> De combien de points en moyenne ? L'effet est-il homogène selon les arrondissements ?


In [9]:
# Distribution statistique par quartiles du taux d'occupation en fonction de la météo
df_q2 = spark.sql("""
-- percentile_approx renvoie des quantiles APPROCHÉS mais calculables en une passe distribuée
-- (un quantile exact exigerait de trier toutes les valeurs). Q1 / médiane / Q3 décrivent
-- toute la distribution, alors que la moyenne reste sensible aux valeurs extrêmes.
SELECT est_pluie,
       COUNT(*)                                       AS nb_snapshots,
       ROUND(AVG(taux_occupation), 4)                 AS taux_moyen,
       ROUND(percentile_approx(taux_occupation, 0.25), 3) AS q1,
       ROUND(percentile_approx(taux_occupation, 0.50), 3) AS mediane,
       ROUND(percentile_approx(taux_occupation, 0.75), 3) AS q3
FROM disponibilite
WHERE est_pluie IS NOT NULL          -- on écarte les relevés sans météo associée
GROUP BY est_pluie
ORDER BY est_pluie
""")
df_q2.show()

# Écart brut (en points de pourcentage) entre relevés pluvieux et secs
moyennes = {r["est_pluie"]: r["taux_moyen"] for r in df_q2.collect()}
print(f"Écart brut pluie - sec : {100 * (moyennes[True] - moyennes[False]):+.2f} points de taux d'occupation")

+---------+------------+----------+-----+-------+-----+
|est_pluie|nb_snapshots|taux_moyen|   q1|mediane|   q3|
+---------+------------+----------+-----+-------+-----+
|    false|    10293254|    0.3671|0.143|  0.306|0.556|
|     true|      476008|    0.3829| 0.16|  0.327|0.576|
+---------+------------+----------+-----+-------+-----+



Écart brut pluie - sec : +1.58 points de taux d'occupation


#### Correction du biais de composition

L'écart brut pluie / sec mélange l'effet de la météo et celui de l'heure. La cellule suivante
le corrige par **standardisation** (comparaison à heure et type de jour égaux).

In [10]:
# L'écart brut ci-dessus est BIAISÉ : la pluie tombe plus à certaines heures et certains jours.
# Si elle tombait surtout la nuit, quand les stations sont plus pleines, j'attribuerais à la
# pluie l'effet de l'heure. Je compare donc pluie et sec À L'INTÉRIEUR de chaque strate
# (heure x jour de semaine / week-end), puis je moyenne les écarts en pondérant par le nombre
# de relevés pluvieux de la strate (standardisation).
spark.sql("""
WITH par_strate AS (
    SELECT heure, est_weekend, est_pluie,
           AVG(taux_occupation) AS taux, COUNT(*) AS n
    FROM disponibilite
    WHERE est_pluie IS NOT NULL
    GROUP BY heure, est_weekend, est_pluie
)
SELECT ROUND(100 * SUM((p.taux - s.taux) * p.n) / SUM(p.n), 3) AS ecart_ajuste_points,
       SUM(p.n)                                                AS nb_releves_pluvieux
FROM par_strate p
JOIN par_strate s
  ON p.heure = s.heure AND p.est_weekend = s.est_weekend
WHERE p.est_pluie AND NOT s.est_pluie
""").show()
# Interprétation : comparer avec l'écart brut. Un signe ou une amplitude différents montrent
# que l'heure / le jour masquaient une partie de l'effet.

+-------------------+-------------------+
|ecart_ajuste_points|nb_releves_pluvieux|
+-------------------+-------------------+
|               1.52|             476008|
+-------------------+-------------------+



In [11]:
# Effet de la pluie par arrondissement
df_q2_arr = spark.sql("""
-- Deux moyennes conditionnelles (CASE WHEN dans AVG) calculées dans le MÊME parcours des
-- données, puis leur différence : delta = taux sous la pluie - taux par temps sec.
SELECT code_arr,
       ROUND(AVG(CASE WHEN est_pluie     THEN taux_occupation END), 4) AS taux_pluie,
       ROUND(AVG(CASE WHEN NOT est_pluie THEN taux_occupation END), 4) AS taux_sec,
       ROUND(AVG(CASE WHEN est_pluie     THEN taux_occupation END)
           - AVG(CASE WHEN NOT est_pluie THEN taux_occupation END), 4) AS delta,
       COUNT(*)                                                        AS nb_snapshots
FROM disponibilite
WHERE est_pluie IS NOT NULL
GROUP BY code_arr
HAVING COUNT(*) > 1000               -- ignore les codes très peu représentés
ORDER BY delta
""")
df_q2_arr.show(30)
# delta = taux(pluie) - taux(sec).
# ATTENTION à l'interprétation : le taux d'occupation mesure un NIVEAU (vélos / places) et
# reste un lien indirect avec le nombre de trajets. La cellule suivante mesure donc aussi un
# indicateur plus direct de l'activité.

+--------+----------+--------+-------+------------+
|code_arr|taux_pluie|taux_sec|  delta|nb_snapshots|
+--------+----------+--------+-------+------------+
|      46|    0.3721|  0.4118|-0.0397|       23603|
|      25|    0.2884|  0.3004| -0.012|       47196|
|      42|    0.3966|  0.4049|-0.0083|      259578|
|      48|    0.1674|  0.1702|-0.0027|       78660|
|       8|    0.3456|  0.3471|-0.0015|      410902|
|      41|    0.3265|  0.3264| 1.0E-4|      204516|
|      44|     0.468|  0.4673| 7.0E-4|      128807|
|      45|    0.4506|  0.4468| 0.0038|       31464|
|       2|    0.4357|  0.4318| 0.0039|      203233|
|      23|     0.369|  0.3642| 0.0048|      165186|
|      27|    0.4441|  0.4388| 0.0053|       39330|
|      47|    0.5929|   0.587| 0.0059|       55062|
|      13|    0.3917|   0.385| 0.0066|      519156|
|       5|    0.4867|  0.4769| 0.0098|      290306|
|      15|    0.4975|  0.4873| 0.0101|      702746|
|      17|     0.293|  0.2816| 0.0114|      469798|
|       9|  

#### Un indicateur plus direct : les mouvements de vélos

Le taux d'occupation ne mesure pas des trajets. On calcule donc aussi le nombre de vélos qui
entrent ou sortent d'une station entre deux relevés consécutifs.

In [12]:
# Proxy plus direct des déplacements : le nombre de vélos qui apparaissent ou disparaissent
# d'une station entre deux relevés consécutifs (~15 min). Moins de trajets => moins de
# mouvements. On utilise LAG (fonction de fenêtrage, détaillée en 1.3) pour comparer chaque
# relevé au précédent DE LA MÊME STATION.
fenetre_mouv = Window.partitionBy("station_id").orderBy("horodatage")

df_mouvements = (
    df.select(
        "station_id", "horodatage", "heure", "est_weekend", "est_pluie",
        (F.col("velos_meca") + F.col("velos_elec")).alias("velos"),
    )
    .withColumn("velos_prec", F.lag("velos").over(fenetre_mouv))
    .withColumn("ts_prec",    F.lag("horodatage").over(fenetre_mouv))
    .withColumn("ecart_min",
        (F.unix_timestamp("horodatage") - F.unix_timestamp("ts_prec")) / 60)
    # On ne compare que des relevés vraiment consécutifs : un trou de plusieurs heures dans la
    # collecte fabriquerait de faux « grands mouvements ».
    .filter("ecart_min > 0 AND ecart_min <= 20 AND est_pluie IS NOT NULL")
    .withColumn("mouvement", F.abs(F.col("velos") - F.col("velos_prec")))
)
df_mouvements.createOrReplaceTempView("mouvements")

spark.sql("""
WITH par_strate AS (
    SELECT heure, est_weekend, est_pluie, AVG(mouvement) AS mouv, COUNT(*) AS n
    FROM mouvements
    GROUP BY heure, est_weekend, est_pluie
)
SELECT ROUND(SUM(s.mouv * p.n) / SUM(p.n), 4)          AS mouvements_sec,
       ROUND(SUM(p.mouv * p.n) / SUM(p.n), 4)          AS mouvements_pluie,
       ROUND(100 * (SUM(p.mouv * p.n) / SUM(s.mouv * p.n) - 1), 2) AS variation_pct
FROM par_strate p
JOIN par_strate s ON p.heure = s.heure AND p.est_weekend = s.est_weekend
WHERE p.est_pluie AND NOT s.est_pluie
""").show()
# Lecture : « variation_pct » < 0 = moins de mouvements de vélos sous la pluie, à heure et type
# de jour égaux, c'est-à-dire moins de trajets. C'est la réponse la plus directe à la question 2.

+--------------+----------------+-------------+
|mouvements_sec|mouvements_pluie|variation_pct|
+--------------+----------------+-------------+
|        0.6733|          0.5049|       -25.01|
+--------------+----------------+-------------+



### Question 3 : Saisonnalité intra-journalière

> Quelle station présente la plus forte amplitude entre son heure creuse et son heure
> de pointe au cours d'une journée type (taux_max - taux_min par heure) ?

C'est un cas d'usage typique des **fonctions de fenêtrage**.


In [13]:
# Quelles sont les 15 stations dont le comportement est le plus pendulaire — presque vides à certaines heures, saturées à d'autres ?
df_q3 = spark.sql("""
-- Deux niveaux d'agrégation emboîtés (CTE) :
--  1. `profil`    : taux moyen de chaque station à chaque heure de la journée (24 points).
--  2. `amplitude` : plus haut et plus bas de ce profil, et à quelles heures.
-- Une station « pendulaire » a un grand écart entre son heure de pointe et son heure creuse.
-- On exige >= 200 relevés au total pour éviter les stations trop peu observées.
WITH profil AS (
    SELECT station_id, nom_station, heure,
           AVG(taux_occupation) AS taux_h, COUNT(*) AS n
    FROM disponibilite
    GROUP BY station_id, nom_station, heure
),
amplitude AS (
    SELECT station_id, nom_station,
           MAX(taux_h)                AS taux_max,
           MIN(taux_h)                AS taux_min,
           MAX(taux_h) - MIN(taux_h)  AS amplitude,
           MAX_BY(heure, taux_h)      AS heure_pointe,   -- heure du taux maximal
           MIN_BY(heure, taux_h)      AS heure_creuse,   -- heure du taux minimal
           SUM(n)                     AS nb_snapshots
    FROM profil
    GROUP BY station_id, nom_station
)
SELECT station_id, nom_station,
       ROUND(taux_min, 3) AS taux_min, ROUND(taux_max, 3) AS taux_max,
       ROUND(amplitude, 3) AS amplitude, heure_creuse, heure_pointe
FROM amplitude
WHERE nb_snapshots >= 200
ORDER BY amplitude DESC
LIMIT 15
""")
df_q3.show(truncate=False)

+----------+------------------------------------+--------+--------+---------+------------+------------+
|station_id|nom_station                         |taux_min|taux_max|amplitude|heure_creuse|heure_pointe|
+----------+------------------------------------+--------+--------+---------+------------+------------+
|1479      |Gare du Nord - Denain               |0.124   |0.861   |0.738    |13          |23          |
|1481      |Gare du Nord - Faubourg Saint-Denis |0.133   |0.824   |0.691    |13          |22          |
|1719      |Léon - Doudeauville                 |0.134   |0.74    |0.606    |12          |22          |
|2020      |Place de la Madeleine - Royale      |0.271   |0.873   |0.601    |2           |15          |
|1500      |Godot de Mauroy - Madeleine         |0.257   |0.856   |0.599    |2           |16          |
|1303      |Danielle Casanova - Place Vendôme   |0.212   |0.798   |0.586    |5           |16          |
|1359      |Erasme - Ulm                        |0.079   |0.662 

In [14]:
# [EXERCICE]
# En utilisant Spark SQL, calculez pour chaque station :
# - le taux d'occupation moyen un jour de semaine sec (est_pluie = false)
# - le taux d'occupation moyen un week-end pluvieux (est_pluie = true)
# - le ratio entre les deux
# Affichez les 10 stations avec le ratio le plus élevé (plus forte différence).
#
# Rappel : est_weekend est un booléen, est_pluie aussi.
# ──────────────────────────────────────────────────────────────────────────

# Votre requête ici :
spark.sql("""
-- Un seul parcours : deux moyennes conditionnelles par station (CASE WHEN dans AVG), puis le
-- ratio calculé dans la requête englobante (on ne peut pas réutiliser un alias dans le même SELECT).
-- Garde-fous : au moins 20 relevés de « week-end pluvieux » (données rares sur 4 mois) et un
-- taux de semaine > 0,05, sinon le ratio explose sur de simples effets de petit échantillon.
SELECT station_id, nom_station,
       ROUND(taux_semaine_sec, 3)              AS taux_semaine_sec,
       ROUND(taux_we_pluie, 3)                 AS taux_we_pluie,
       ROUND(taux_we_pluie / taux_semaine_sec, 3) AS ratio,
       n_we_pluie
FROM (
    SELECT station_id, nom_station,
           AVG(CASE WHEN NOT est_weekend AND NOT est_pluie THEN taux_occupation END) AS taux_semaine_sec,
           AVG(CASE WHEN est_weekend AND est_pluie         THEN taux_occupation END) AS taux_we_pluie,
           COUNT(CASE WHEN est_weekend AND est_pluie       THEN 1 END)               AS n_we_pluie
    FROM disponibilite
    GROUP BY station_id, nom_station
)
WHERE n_we_pluie >= 20 AND taux_semaine_sec > 0.05
ORDER BY ratio DESC
LIMIT 10
""").show(truncate=False)

+----------+-------------------------------------+----------------+-------------+-----+----------+
|station_id|nom_station                          |taux_semaine_sec|taux_we_pluie|ratio|n_we_pluie|
+----------+-------------------------------------+----------------+-------------+-----+----------+
|2017      |Place de la Division Leclerc         |0.093           |0.313        |3.351|88        |
|1251      |Cimetière de Montmartre              |0.139           |0.392        |2.814|88        |
|1344      |Edouard Pailleron - Bouret           |0.226           |0.552        |2.439|88        |
|1937      |Pierre et Marie Curie - Julien Grimau|0.139           |0.335        |2.419|88        |
|2214      |Saint-Fargeau - Mortier              |0.139           |0.333        |2.395|88        |
|1411      |Frères Flavien - Porte des Lilas     |0.143           |0.335        |2.345|88        |
|1433      |Gare Montparnasse - Vaugirard        |0.146           |0.33         |2.257|88        |
|1430     

---
## 1.3 Fonctions de fenêtrage analytique

Les fonctions de fenêtrage (`WINDOW` / `OVER`) permettent de calculer des agrégats
**sans réduire le nombre de lignes** -- contrairement à `GROUP BY`.
Elles sont indispensables pour les analyses de séries temporelles.

### Les trois familles de fonctions fenêtrées

```
Ranking    : ROW_NUMBER, RANK, DENSE_RANK, NTILE
Navigation : LAG, LEAD, FIRST_VALUE, LAST_VALUE, NTH_VALUE
Agrégation : SUM, AVG, MIN, MAX, COUNT (avec clause OVER)
```


In [15]:
# Cas concret : pour chaque station, calculer le taux d'occupation
# de la fenêtre précédente (LAG) et suivante (LEAD),
# ainsi qu'une moyenne mobile sur 3 snapshots.
station_cible = 1042   # à adapter selon vos données

# Une fenêtre = (partitionBy : les groupes indépendants) + (orderBy : l'ordre dans le groupe).
# Ici chaque station forme sa propre série chronologique.
fenetre_station = Window.partitionBy("station_id").orderBy("horodatage")

df_avec_lag = (
    df
    # On filtre AVANT la fenêtre : Spark n'a alors à trier qu'une seule série au lieu de 1 400.
    .filter(F.col("station_id") == station_cible)
    .select("station_id", "horodatage", "taux_occupation")
    .withColumn("taux_precedent", F.lag("taux_occupation", 1).over(fenetre_station))   # relevé d'avant
    .withColumn("taux_suivant",   F.lead("taux_occupation", 1).over(fenetre_station))  # relevé d'après
    # rowsBetween(-2, 0) : la ligne courante et les deux précédentes = moyenne mobile sur 3 relevés.
    .withColumn("moyenne_mobile_3",
        F.round(F.avg("taux_occupation").over(fenetre_station.rowsBetween(-2, 0)), 4))
)
df_avec_lag.show(truncate=False)

+----------+-------------------+---------------+--------------+------------+----------------+
|station_id|horodatage         |taux_occupation|taux_precedent|taux_suivant|moyenne_mobile_3|
+----------+-------------------+---------------+--------------+------------+----------------+
|1042      |2020-11-26 12:59:00|0.2273         |NULL          |0.2273      |0.2273          |
|1042      |2020-11-26 13:06:00|0.2273         |0.2273        |0.2273      |0.2273          |
|1042      |2020-11-26 13:21:00|0.2273         |0.2273        |0.2727      |0.2273          |
|1042      |2020-11-26 13:32:00|0.2727         |0.2273        |0.2727      |0.2424          |
|1042      |2020-11-26 13:47:00|0.2727         |0.2727        |0.3182      |0.2576          |
|1042      |2020-11-26 14:25:00|0.3182         |0.2727        |0.3636      |0.2879          |
|1042      |2020-11-26 14:32:00|0.3636         |0.3182        |0.3636      |0.3182          |
|1042      |2020-11-26 14:47:00|0.3636         |0.3636      

In [16]:
# Classement des stations par taux d'occupation moyen, à chaque heure de la journée
# ROW_NUMBER() numérote les lignes dans chaque partition (ici : chaque heure)
# Je partitionne la fenêtre par HEURE et je l'ordonne par taux décroissant : row_number()
# donne alors le rang de chaque station dans le classement de son heure.
fenetre_heure = Window.partitionBy("heure").orderBy(F.desc("taux_moyen"))

df_rank = (
    df
    .groupBy("heure", "station_id", "nom_station")
    .agg(F.round(F.avg("taux_occupation"), 4).alias("taux_moyen"))   # 1) moyenne par (heure, station)
    .withColumn("rang", F.row_number().over(fenetre_heure))          # 2) classement dans l'heure
    .filter(F.col("rang") <= 2)                                      # 3) on garde le podium (2 par heure)
    .orderBy("heure", "rang")
)
df_rank.show(48, truncate=False)

+-----+----------+----------------------------------+----------+----+
|heure|station_id|nom_station                       |taux_moyen|rang|
+-----+----------+----------------------------------+----------+----+
|0    |1511      |Grenelle - Dr Finlay              |0.8702    |1   |
|0    |2008      |Place de Verdun                   |0.852     |2   |
|1    |1511      |Grenelle - Dr Finlay              |0.8667    |1   |
|1    |2008      |Place de Verdun                   |0.8215    |2   |
|2    |2008      |Place de Verdun                   |0.8654    |1   |
|2    |1511      |Grenelle - Dr Finlay              |0.8578    |2   |
|3    |2008      |Place de Verdun                   |0.8567    |1   |
|3    |1511      |Grenelle - Dr Finlay              |0.851     |2   |
|4    |2008      |Place de Verdun                   |0.857     |1   |
|4    |1511      |Grenelle - Dr Finlay              |0.8287    |2   |
|5    |2008      |Place de Verdun                   |0.846     |1   |
|5    |1511      |Gr

In [17]:
# Calcul de la variation du taux sur une heure glissante
# UNBOUNDED PRECEDING -> ligne actuelle = cumul depuis le début de la partition

fenetre_cumul = (
    Window
    .partitionBy("station_id")
    .orderBy("horodatage")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Variation par rapport au snapshot précédent (delta instantané)
station_cible = 1042   # à adapter selon vos données

df_delta = (
    df
    .filter(F.col("station_id") == station_cible)
    .select("station_id", "horodatage", "taux_occupation")
    .withColumn("variation",                                   # delta instantané : taux(t) - taux(t-1)
        F.round(F.col("taux_occupation")
                - F.lag("taux_occupation").over(fenetre_station), 4))
    .withColumn("variation_cumulee",                           # somme des variations depuis le début
        F.round(F.sum("variation").over(fenetre_cumul), 4))
)
df_delta.show(truncate=False)
# Sanity check : variation_cumulee = taux(t) - taux(premier relevé) (les variations se télescopent).

+----------+-------------------+---------------+---------+-----------------+
|station_id|horodatage         |taux_occupation|variation|variation_cumulee|
+----------+-------------------+---------------+---------+-----------------+
|1042      |2020-11-26 12:59:00|0.2273         |NULL     |NULL             |
|1042      |2020-11-26 13:06:00|0.2273         |0.0      |0.0              |
|1042      |2020-11-26 13:21:00|0.2273         |0.0      |0.0              |
|1042      |2020-11-26 13:32:00|0.2727         |0.0454   |0.0454           |
|1042      |2020-11-26 13:47:00|0.2727         |0.0      |0.0454           |
|1042      |2020-11-26 14:25:00|0.3182         |0.0455   |0.0909           |
|1042      |2020-11-26 14:32:00|0.3636         |0.0454   |0.1363           |
|1042      |2020-11-26 14:47:00|0.3636         |0.0      |0.1363           |
|1042      |2020-11-26 15:06:00|0.3182         |-0.0454  |0.0909           |
|1042      |2020-11-26 15:25:00|0.2273         |-0.0909  |0.0              |

---
## 1.4 Delta Lake : transactions, time-travel et MERGE

Delta Lake est une couche de stockage transactionnel construite par-dessus Parquet.
Elle apporte à Spark les propriétés **ACID** qui manquent au Parquet brut :

| Propriété | Parquet brut | Delta Lake |
|-----------|-------------|------------|
| Lecture cohérente pendant une écriture | Non | Oui (MVCC) |
| Annulation d'une écriture partielle | Non | Oui |
| Historique des versions | Non | Oui (time-travel) |
| Mise à jour / suppression de lignes | Non | Oui (MERGE, UPDATE, DELETE) |
| Optimisation automatique | Non | Oui (OPTIMIZE, Z-ORDER) |

### Écriture en format Delta


In [18]:
from delta.tables import DeltaTable

# ADAPTATION : l'énoncé écrit 2022 puis ajoute 2023. Nos données ne couvrent qu'un hiver :
# on écrit d'abord novembre 2020 -> février 2021 (« historique »), puis on AJOUTE mars -> avril
# 2021 (« nouveau batch »), ce qui crée bien deux versions successives de la table.
FIN_PERIODE_1 = 202102          # AAAAMM : dernière année-mois de la 1re écriture
cle_mois = F.col("annee") * 100 + F.col("mois")

df_periode1 = df.filter(cle_mois <= FIN_PERIODE_1)

t0 = time.perf_counter()
(
    df_periode1
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Écriture période 1 : {time.perf_counter()-t0:.1f} s  --  {df_periode1.count():,} lignes")

# Vérification de la structure Delta
import os
fichiers_delta = list(Path(DELTA_DISPONIBLE).rglob("*.parquet"))
log_delta      = list(Path(DELTA_DISPONIBLE / "_delta_log").glob("*.json"))
print(f"Fichiers Parquet : {len(fichiers_delta)}")
print(f"Entrées dans le transaction log : {len(log_delta)}")

Écriture période 1 : 5.4 s  --  6,967,885 lignes
Fichiers Parquet : 27
Entrées dans le transaction log : 4


In [19]:
# Ajout de la période 2 (mars-avril 2021) -- mode "append"
df_periode2 = df.filter(cle_mois > FIN_PERIODE_1)

t0 = time.perf_counter()
(
    df_periode2 # Enregistrer les données au format DeltaLake, en mode ajout
    .write
    .format("delta")
    .mode("append")               # ajoute des fichiers sans toucher aux précédents ;
    .partitionBy("annee", "mois")  # une NOUVELLE version apparaît dans le transaction log
    .save(str(DELTA_DISPONIBLE))
)
print(f"Ajout période 2 : {time.perf_counter()-t0:.1f} s  --  {df_periode2.count():,} lignes")

# Historique des versions : chaque opération crée une nouvelle version
delta_table = DeltaTable.forPath(spark, str(DELTA_DISPONIBLE))
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

Ajout période 2 : 2.6 s  --  3,801,377 lignes
+-------+-----------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                                                                                                                                                                                                        |
+-------+-----------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|4      |2026-09-21 22:41:36.913|WRITE    |{mode -> Append, partitionBy -> ["annee","mois"]}                      

### Time-travel : interroger une version passée

Delta Lake conserve toutes les versions de la table dans le transaction log.
On peut interroger n'importe quelle version passée avec la clause
`VERSION AS OF` ou `TIMESTAMP AS OF`.


In [20]:
# Lecture de la version 0 (uniquement la période 1 : novembre 2020 -> février 2021)
# option("versionAsOf", 0) fait lire à Delta l'état de la table tel qu'il était après la
# 1re transaction : Delta ignore les fichiers de l'append (il ne les supprime pas).
df_v0 = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .load(str(DELTA_DISPONIBLE))
)
print(f"Version 0 (période 1 uniquement) : {df_v0.count():,} lignes")

# Lecture de la version courante
df_current = spark.read.format("delta").load(str(DELTA_DISPONIBLE))
print(f"Version courante (périodes 1+2) : {df_current.count():,} lignes")

# Enregistrement comme vue SQL pour la suite
df_current.createOrReplaceTempView("disponibilite_delta")

Version 0 (période 1 uniquement) : 6,967,885 lignes


Version courante (périodes 1+2) : 10,769,262 lignes


### `MERGE INTO` : mise à jour incrémentale

`MERGE INTO` est l'opération la plus puissante de Delta Lake. Elle permet de
**synchroniser** une table cible avec une table source en une seule passe :
insertions des nouvelles lignes, mises à jour des lignes existantes,
suppressions optionnelles.

Cas d'usage typique : arrivée quotidienne d'un nouveau batch de snapshots.


In [21]:
# Simulation : un nouveau batch arrive avec des corrections
# (quelques lignes modifiées + quelques nouvelles lignes)
from pyspark.sql.functions import lit, current_timestamp

# On prend 500 snapshots existants et on simule une correction du taux_occupation.
# dropDuplicates sur la clé du MERGE garantit qu'une même (station, instant) n'apparaît qu'une
# fois dans le batch (sinon le MERGE échouerait : plusieurs sources pour une cible).
# .cache() fige l'échantillon : sans lui, limit() peut renvoyer d'autres lignes à chaque
# recalcul, et le batch que je compte ne serait plus celui que je fusionne.
df_corrections = (
    df_current
    .filter(F.col("mois") == 1)
    .dropDuplicates(["station_id", "horodatage"])
    .limit(500)
    .withColumn("taux_occupation", F.round(F.col("taux_occupation") * 0.98, 4))
    .withColumn("source", lit("correction_batch"))
    .cache()
)

# Quelques nouvelles lignes fictives (snapshots manqués)
# ADAPTATION : je décale d'UN an (le modèle en décalait 2) : les relevés tombent en 2022, année
# absente de la table, donc sans correspondance, et le MERGE doit les INSÉRER.
df_nouveaux = (
    df_current
    .filter(F.col("mois") == 1)
    .dropDuplicates(["station_id", "horodatage"])
    .limit(50)
    .withColumn("horodatage",
        F.col("horodatage") + F.expr("INTERVAL 1 YEAR"))
    .withColumn("annee", lit(2022))
    .withColumn("source", lit("nouveau_batch"))
    .cache()
)

df_batch = df_corrections.union(df_nouveaux).drop("source")
print(f"Batch entrant : {df_batch.count()} lignes ({df_corrections.count()} corrections + {df_nouveaux.count()} nouvelles)")

Batch entrant : 550 lignes (500 corrections + 50 nouvelles)


In [22]:
# MERGE INTO : upsert (update + insert)
# Le MERGE compare chaque ligne du batch à la table selon la condition ci-dessous, puis :
#  - si elle existe déjà (même station, même instant) -> on met à jour toutes les colonnes ;
#  - sinon -> on l'insère.
# Le tout est UNE transaction atomique : un échec à mi-chemin ne laisse aucun état intermédiaire.
(
    delta_table.alias("cible")
    .merge(
        df_batch.alias("source"),
        # Condition de correspondance : même station, même horodatage
        "cible.station_id = source.station_id AND cible.horodatage = source.horodatage"
    )
    .whenMatchedUpdateAll()     # si correspondance : on écrase toutes les colonnes
    .whenNotMatchedInsertAll()  # si pas de correspondance : on insère
    .execute()
)

# Vérification : la table a une nouvelle version
delta_table.history().select(
    "version", "timestamp", "operation",
    "operationMetrics"
).show(5, truncate=False)

+-------+-----------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                                              

In [23]:
# [EXERCICE]
# Le MERGE précédent a mis à jour des lignes existantes.
# Utilisez le time-travel pour comparer le taux_occupation moyen
# de janvier 2022 AVANT et APRÈS le merge.
#
# Indice : lisez la version 1 (avant merge) et la version courante,
# puis comparez avec une agrégation.
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
# ADAPTATION : nos données vont de nov. 2020 à avril 2021 : on compare janvier 2021.
# Versions de la table : 0 = écriture initiale, 1 = ajout (avant merge), 2 = merge.
def taux_moyen_janvier(version: int | None = None) -> float:
    """Taux d'occupation moyen de janvier 2021 dans une version de la table Delta.

    Args:
        version: Numéro de version Delta, ou None pour la version courante.

    Returns:
        Le taux d'occupation moyen (float).
    """
    lecteur = spark.read.format("delta")
    if version is not None:
        lecteur = lecteur.option("versionAsOf", version)
    return (
        lecteur.load(str(DELTA_DISPONIBLE))
        .filter("annee = 2021 AND mois = 1")
        .agg(F.avg("taux_occupation"))
        .first()[0]
    )

avant = taux_moyen_janvier(version=1)   # état de la table AVANT le merge
apres = taux_moyen_janvier()            # état courant, après le merge
print(f"Taux moyen janvier 2021 avant merge : {avant:.8f}")
print(f"Taux moyen janvier 2021 après merge : {apres:.8f}")
print(f"Différence                          : {apres - avant:+.8f}")
# La différence est infime : 500 lignes corrigées (x 0,98) sur plus d'un million de relevés
# de janvier. Le time-travel permet précisément de le mesurer, alors que sans versionnage,
# l'état antérieur serait perdu.

Taux moyen janvier 2021 avant merge : 0.38054232
Taux moyen janvier 2021 après merge : 0.38054083
Différence                          : -0.00000149
